### Params

In [51]:
seq = "123"

In [52]:
!ls -l outputs/full_split/splatad/

total 40
drwxr-xr-x 3 jovyan jovyan 4096 Sep 28 17:09 2025-09-28_170104
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:08 2025-10-02_160038
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:14 2025-10-02_160936
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:15 2025-10-02_161154
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:20 2025-10-02_161658
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:23 2025-10-02_161940
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:27 2025-10-02_162120
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:28 2025-10-02_162326
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 16:41 2025-10-02_163623
drwxr-xr-x 3 jovyan jovyan 4096 Oct  2 17:13 2025-10-02_170733


In [53]:
seq2folder = {
    "001": "2025-09-28_170104",
    "011": "2025-10-02_160038",
    "016": "2025-10-02_160936",
    "028": "2025-10-02_161154",
    "053": "2025-10-02_161658", 
    # "063": "2025-10-02_161940",
    # "084": "2025-10-02_162120",
    "106": "2025-10-02_162326",
    "123": "2025-10-02_163623",
    "158": "2025-10-02_170733"
}

In [54]:
folder = seq2folder[seq]
cfg = f"outputs/full_split/splatad/{folder}/config.yml"
shift = [-3.0, 0.0, 0.0]

str_shift = f"{shift[0]} {shift[1]} {shift[2]}"

### Rendering

In [107]:
!python nerfstudio/scripts/render_shifted.py --load-config {cfg} --output_path shifted/{seq} --shift {str_shift} --render_point_clouds False

2025-10-05 20:23:53.131949: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-05 20:23:53.276678: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-05 20:23:54.642493: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/home/jovyan/users/npata

### Video check

In [7]:
selected_cameras = ["left_camera", "front_camera"]
base_folder = f"shifted/{seq}/test"
fps = 10

In [8]:
import imageio
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
import glob
import os
import numpy as np

def create_video_with_imageio(base_folder, camera_name, output_path, fps=10, reverse=False):
    # Get image paths
    if camera_name == "lidar":
        rgb_folder = Path(base_folder)
        gt_rgb_folder = Path(base_folder)
        rgb_camera_path = os.path.join(rgb_folder, "lidar_vis" )
        gt_rgb_camera_path = os.path.join(gt_rgb_folder, "lidar_vis_gt")
        rgb_images = sorted(glob.glob(os.path.join(rgb_camera_path, "*.png")))
        gt_rgb_images = sorted(glob.glob(os.path.join(gt_rgb_camera_path, "*.png")))
    else:
        rgb_folder = Path(base_folder) / "rgb" 
        gt_rgb_folder = Path(base_folder) / "gt-rgb"
        rgb_camera_path = os.path.join(rgb_folder, camera_name)
        gt_rgb_camera_path = os.path.join(gt_rgb_folder, camera_name)
        rgb_images = sorted(glob.glob(os.path.join(rgb_camera_path, "*.jpg")))
        gt_rgb_images = sorted(glob.glob(os.path.join(gt_rgb_camera_path, "*.jpg")))
        
    frames = []
    for i, (rgb_path, gt_rgb_path) in enumerate(zip(rgb_images, gt_rgb_images)):
        # Read images using PIL
        rgb_img = Image.open(rgb_path)
        gt_rgb_img = Image.open(gt_rgb_path)
        
        # Get dimensions
        w, h = rgb_img.size

        w = int(w * 0.4)
        h = int(h * 0.4)
        
        # Resize GT image to match RGB if needed
        gt_rgb_img = gt_rgb_img.resize((w, h))
        rgb_img = rgb_img.resize((w, h))
        
        # Create side-by-side image
        combined_img = Image.new('RGB', (w * 2, h))
        combined_img.paste(rgb_img, (0, 0))
        combined_img.paste(gt_rgb_img, (w, 0))
        
        # Add text labels using PIL
        draw = ImageDraw.Draw(combined_img)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        except:
            font = ImageFont.load_default()
        
        # Add labels
        if camera_name != "lidar":
            if reverse:
                draw.text((10, 10), f'Reverse shift', fill=(255, 255, 255), font=font)
                draw.text((w + 10, 10), f'Ground Truth with shift {shift}', fill=(255, 255, 255), font=font)
            else:
                draw.text((10, 10), f'Shift {shift}', fill=(255, 255, 255), font=font)
                draw.text((w + 10, 10), 'Ground Truth', fill=(255, 255, 255), font=font)

        frame_array = np.array(combined_img)
        frames.append(frame_array)
    imageio.mimsave(output_path, frames, fps=fps)

In [110]:
(Path(base_folder) / "videos").mkdir(parents=True, exist_ok=True)

for selected_camera in selected_cameras:
    output_video_path = Path(base_folder) / "videos" / f"{selected_camera}.mp4"
    create_video_with_imageio(base_folder, selected_camera, output_video_path, fps)

### Final dataset

In [5]:
!CUDA_VISIBLE_DEVICES=3 python nerfstudio/scripts/render_shifted.py --load-config {cfg} --output_path shifted/{seq} --shift {str_shift} --render_point_clouds True

2025-10-10 02:58:52.880286: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-10 02:58:53.074187: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-10 02:58:54.602651: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/cuda-12.4/lib

In [6]:
!cp -r /workspace/datasets/self-driving/pandaset/{seq} data/pandaset/

In [7]:
!cp -r shifted/{seq}/test/lidar/* data/pandaset/{seq}/lidar

In [8]:
!cp -r shifted/{seq}/test/rgb/front_camera/* data/pandaset/{seq}/camera/front_camera

In [9]:
!cp -r shifted/{seq}/test/rgb/left_camera/* data/pandaset/{seq}/camera/left_camera

In [10]:
!cp -r shifted/{seq}/test/rgb/right_camera/* data/pandaset/{seq}/camera/right_camera

In [11]:
!cp -r shifted/{seq}/test/rgb/back_camera/* data/pandaset/{seq}/camera/back_camera

In [12]:
!cp -r shifted/{seq}/test/rgb/front_left_camera/* data/pandaset/{seq}/camera/front_left_camera

In [13]:
!cp -r shifted/{seq}/test/rgb/front_right_camera/* data/pandaset/{seq}/camera/front_right_camera

### Optimize

In [22]:
#CUDA_VISIBLE_DEVICES=0 python nerfstudio/scripts/train.py splatad --experiment_name="shifts" pandaset-data --sequence {seq}

### Test

In [55]:
!ls -l outputs/cycle/splatad/

total 44
drwxr-xr-x 4 jovyan jovyan 4096 Oct 11 01:47 2025-10-06_002038
drwxr-xr-x 4 jovyan jovyan 4096 Oct 11 01:53 2025-10-06_011816
drwxr-xr-x 4 jovyan jovyan 4096 Oct 11 01:59 2025-10-06_012220
drwxr-xr-x 4 jovyan jovyan 4096 Oct 11 01:28 2025-10-06_012647
drwxr-xr-x 2 jovyan jovyan 4096 Oct  6 01:36 2025-10-06_013647
drwxr-xr-x 3 jovyan jovyan 4096 Oct  6 01:44 2025-10-06_014016
drwxr-xr-x 4 jovyan jovyan 4096 Oct 11 02:26 2025-10-06_014405
drwxr-xr-x 4 jovyan jovyan 4096 Oct 11 02:28 2025-10-06_014531
drwxr-xr-x 4 jovyan jovyan 4096 Oct 11 02:17 2025-10-07_230441
drwxr-xr-x 2 jovyan jovyan 4096 Oct 10 03:07 2025-10-10_030721
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 03:16 2025-10-10_031201


* 2025-10-06_002038 -- 001
* 2025-10-06_012220 -- 016
* 2025-10-06_011816 -- 011
* 2025-10-06_012647 -- 053
* 2025-10-06_014016 -- 063
* 2025-10-06_014531 -- 123
* 2025-10-06_014405 -- 106
* 2025-10-07_230441 -- 028

In [56]:
str_shift = f"{-shift[0]} {-shift[1]} {-shift[2]}"

folder = "2025-10-06_014531"
cfg = f"outputs/cycle/splatad/{folder}/config.yml"
seq

'123'

In [16]:
!python nerfstudio/scripts/render_shifted.py --load-config {cfg} --output_path reverse_shifted/{seq} --shift {str_shift} --render_point_clouds False --data data/pandaset

2025-10-09 18:30:09.313939: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-09 18:30:09.488192: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-09 18:30:10.988332: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/cuda-12.4/lib

In [17]:
base_folder = f"reverse_shifted/{seq}/test"
(Path(base_folder) / "videos").mkdir(parents=True, exist_ok=True)

print(base_folder)
for selected_camera in selected_cameras:
    output_video_path = Path(base_folder) / "videos" / f"{selected_camera}.mp4"
    create_video_with_imageio(base_folder, selected_camera, output_video_path, fps, reverse=True)

NameError: name 'Path' is not defined

In [ ]:
base_folder = f"reverse_shifted/{seq}/test"
(Path(base_folder) / "videos").mkdir(parents=True, exist_ok=True)

output_video_path = Path(base_folder) / "videos" / f"lidar_new.mp4"
create_video_with_imageio(base_folder, "lidar", output_video_path, fps, reverse=True)

### Compute metrics

In [57]:
!CUDA_VISIBLE_DEVICES=5 python nerfstudio/scripts/eval.py --load-config {cfg} --data-root-path /workspace/datasets/self-driving/pandaset --output-path metrics/cycle_pandaset/splatad/{seq}.json

2025-10-11 02:34:48.063592: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-11 02:34:48.225308: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-11 02:34:49.588921: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda-12.4/lib64:
2025-10-11 

In [5]:
!CUDA_VISIBLE_DEVICES=1 python nerfstudio/scripts/eval.py --load-config outputs/full_split/splatad/2025-10-02_163623/config.yml --data-root-path /workspace/datasets/self-driving/pandaset --output-path metrics/full_pandaset/splatad/123.json

2025-10-08 01:10:17.126818: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-08 01:10:17.291011: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-08 01:10:19.046388: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/jovyan/users/npatakin/libs/cuda-1

In [ ]:
!